# Introduction to Probabilistic Programming

Source:<P>

https://ayandas.me/blog-tut/2020/05/05/probabilistic-programming.html<P>
https://gist.github.com/dasayan05/aca3352cd00058511e8372912ff685d8<P>

Adapted:<P>

Antonio Esteves @ UMinho, Mar 2024

The idea of Probabilistic Programming has long been discussed in the ML literature and got enriched over time. Probabilistic programming is not about writing traditional programs, rather it is building Probabilistic Graphical Models (PGMs) in a imperative programming style, i.e., using iterations, branching, recursion, etc. Just like Automatic Differentiation allows to compute derivatives of arbitrary computation graphs, Black-box methods have been developed to "solve" probabilistic programs. In this notebook, we will provide a generic view on why such a language is indeed possible and how such black-box solvers are materialized. At the end, we will also introduce the Pyro Universal Probabilistic Programming Language.

## Overview

Before we dive into the details, we will make the big picture clear. It is highly advisable to read a good reference about PGMs before proceeding.

### Generative view an execution trace

Probabilistic Programming is NOT really what we usually think of programming, i.e., running completely deterministic hard-coded instructions, which do exactly what they were told to do and nothing more. Rather it is about building PGMs that model our belief about the data generation process. We, as users of such language, would like to express a model in a imperative form, with the objective of encoding all our uncertainty in the way we want. Here is a simple example:

```Python
def model(theta):
    A = Bernoulli([-1, 1] ; theta)
    P = 2 * A
    if A == -1:
        B = Uniform(P, 0)
    else:
        B = Uniform(0, P)
    C = Normal(B, 1)
    return A, P, B, C
```

This is an example of a probabilistic program, where the traditional programming variables became random variables that have uncertainty associated with them, in the form of probability distributions. The elements present in this code are:

* Probability distributions, in this case Normal, Bernoulli, and Uniform.
* Deterministic computations, such as $P=2∗A$.
* Conditioning a random variable with the value of another one. For example, $C \mid B \sim \mathbb{N}(B,1)$.
* Imperative style branching, which provides the model with a dynamic structure.
  
Next figure contains a graphical representation of the model defined by the above program.

<img src="../fig/pyro_graph_model1.png" width="600px" /><P>

Just like the invocation of a traditional compiler on a traditional program produces the desired output, this probabilistic program can be executed through sampling. Imagine that we can execute the program 5 times, or saying in other form, consider that we sample from the model 5 times, and each time we get samples from all the random variables. Each "forward" run is often called an **execution trace** of the model.

```Python
for _ in range(5):
   print(model(0.5))

( 1.000,  2.000,  0.318, -0.069)
(-1.000, -2.000, -1.156, -2.822)
( 1.000,  2.000,  0.594,  0.865)
( 1.000,  2.000,  1.100,  1.079)
(-1.000, -2.000, -0.262, -0.403)
```

This is the so called **generative view** of a model. We typically use the leaf-nodes of PGMs as our observed data. And the rest of the graph can be the **latent factors** of the model, which we either know or want to estimate. In general, a practical PGM can often be encapsulated as a set of latent nodes $Z \triangleq \{Z_1,Z_2,\dots,Z_H\}$ and visible nodes $X \triangleq \{X_1,X_2,\dots,X_V\}$ related probabilistically as $Z \rightarrow X$.

### Training and Inference

From now on, we will use the general notation rather than the specific example. The model may be parametric. For example, we had the Bernoulli success probability $\theta$ in our simple example. The full joint probability is given as $P_\theta (Z,X) = P_\theta (Z) \cdot P_\theta (X \mid Z)$. We would like to do two things:

1. Estimate the model parameters $\theta$ from data.
2. Compute the posterior, i.e., infer latent variables given the data.
 
Both of these tasks are infeasible due to the fact that (i) log-likelihood maximization is not possible due to the presence of latent variables and (ii) for continuous distributions on latent variables, the posterior is intractable.

The way to get around these obstacles is to employ Variational Inference and maximize the Evidence Lower BOund (ELBO) loss, in order to estimate the model parameters and also a set of variational parameters which help building a proxy for the original posterior 
$P_\theta (Z \mid X)$. Mathematically, we choose a known and tractable family of distributions $Q_\phi (Z)$, parameterized by variational parameters $\phi$, to approximate the posterior. During the learning process we maximize the ELBO:

$ELBO(\theta,\phi) \triangleq \mathbb{E}_{Q_\phi} [\log P_\theta(Z,X)-\log Q_\phi(Z)]$

by estimating the gradients with respect to all of its parameters:<P>

\begin{equation}
\nabla_{[\theta,\phi]} ELBO(\theta,\phi)
\tag{1}
\end{equation}

## Black-Box Variational Inference

We want to establish a probabilistic programming setup to optimize the ELBO for any given problem. The problem is formulated by:

1. A model specification $P_\theta(Z,X)$ written in a probabilistic language.
2. An optional parameterized variational model $Q_\phi(Z)$, known as "guide" in Pyro.
3. The observed data $D$.

But, how do we compute the gradients of ELBO, defined in (1)? The problematic gradients are those taken in relation to $\phi$, since $\phi$ also appears in the expectation. To mitigate this problem, we make use of the famous trick known as the **log-derivative** trick, also known as **REINFORCE**. For notational simplicy let us make the following replacement $f(Z,X\ ;\ \theta,\phi) \triangleq \log P_\theta(Z,X)-\log Q_\phi(Z)$ and continue from (1).

\begin{align}
\begin{aligned}
\sum_{\mathbf{Z}} \nabla_{[\theta, \phi]} \bigg[ Q_{\phi}(\mathbf{Z}) \cdot 
    f(\mathbf{Z}, \mathbf{X}; \theta, \phi) \bigg] &= 
    \sum_{\mathbf{Z}} \bigg[ \nabla_{\phi} Q_{\phi}(\mathbf{Z}) 
    \cdot f(\mathbf{Z}, \mathbf{X}; \theta, \phi) + 
    Q_{\phi}(\mathbf{Z})\cdot \nabla_{[\theta, \phi]}
    f(\mathbf{Z}, \mathbf{X}; \theta, \phi) \bigg] \\
&= \sum_{\mathbf{Z}} \bigg[ {\color{red}Q_{\phi}(\mathbf{Z})} \cdot 
    \frac{\nabla_{\phi} Q_{\phi}(\mathbf{Z})}{{\color{red} Q_{\phi}(\mathbf{Z})}} \cdot 
    f(\mathbf{Z}, \mathbf{X}; \theta, \phi) + Q_{\phi}(\mathbf{Z}) \cdot \nabla_{[\theta, \phi]}
    f(\mathbf{Z}, \mathbf{X}; \theta, \phi) \bigg] \\
&= \sum_{\mathbf{Z}} Q_{\phi}(\mathbf{Z}) \cdot \bigg[ 
    {\color{red} \nabla_{\phi} \log Q_{\phi}(\mathbf{Z})} \cdot 
    f(\mathbf{Z}, \mathbf{X}; \theta, \phi) + \nabla_{[\theta, \phi]}
    f(\mathbf{Z}, \mathbf{X}; \theta, \phi) \bigg] \\
&= \mathbb{E}_{Q_{\phi}} \bigg[ \nabla_{[\theta, \phi]} 
    \bigg( \underbrace{\log Q_{\phi}(\mathbf{Z}) \cdot 
    \overline{f(\mathbf{Z}, \mathbf{X}; \theta, \phi)} + 
    f(\mathbf{Z}, \mathbf{X}; \theta, \phi)}_\text{surrogate objective} \bigg) \bigg]
\end{aligned}
\tag{2}
\end{align}
 
Equation (2) shows that the trick helped the $\nabla_{[\theta,\phi]}$ to move inside the $\mathbb{E}[\cdot]$, but as a consequence of this modification, the original $f$ was altered to a surrogate function $f_{surr} \triangleq \bar{f} \cdot \log Q + f$, where the bar protects a quantity from differentiation. Equation (2) is all we need, it provides an insight on how to make the gradient estimation practical. In fact, it can be proven theoretically that this gradient is an unbiased estimate of the true gradient proposed in Equation (1).

Succinctly, we run the guide $L$ times to record a set of $L$ execution-traces, i.e., samples $\hat{Z} \sim Q_\phi$ and compute the following Monte-Carlo approximation to Equation (2):

\begin{equation}
\nabla_{[\theta, \phi]} \mathrm{ELBO}(\theta, \phi) \approx \frac{1}{L} \sum_{\mathbf{\widehat{Z}}\sim\mathbb{Q}_{\phi}} \left[ \nabla_{[\theta, \phi]} f_{surr}(\mathbf{\widehat{Z}}, \mathcal{D}) \right]_{\theta=\theta_{old}, \phi=\phi_{old}}
\tag{3}
\end{equation}
 
The nice thing about Equation (2), or equivalently Equation (3), is that we get the differentiation operator right on top of a deterministic function ($f_{surr}$). It means we can construct $f_{surr}$ as a computation graph and take advantage of automatic differentiation engines. Here is how the computation graph and the graphical model are connected:

<img src="../fig/pyro_graph_model2.png" width="600px" /><P>

Now, we look at the function $f_{surr}$, which is basically built with the log-density terms $\log P_\theta(Z,X)$ and $\log Q_\phi (Z)$. We need a way to compute them flexibly. Please remember that the model and guide are written in a language and hence we have access to their graph-structure. A clever software implementation can harness this structure to estimate the log-densities and eventually $f_{surr}$.

We claimed before that the gradient estimates are unbiased. However, such generic way of computing the gradient introduces high variance in the estimate and make things unstable for complex models. There are a few tricks used widely to get around them. But please note that such tricks always exploit the model-specific structure. Three of such tricks are presented next.

### I. Re-parameterization

We might be lucky and $Q_\phi(Z)$ be re-parameterizable. What that means is the expectation with respect to $Q_\phi(Z)$ can be made free of its parameters and by doing so the gradient operator can be pushed inside without going through the log-derivative trick. So, let us step back a bit and consider the original ELBO gradient in (1). Assuming the re-parameterizable nature, the following can be done:

\begin{align}
\tag{4}
\nabla_{[\theta, \phi]} \mathbb{E}_{Q_{\phi}} \bigg[\log P_{\theta}(\mathbf{Z}, \mathbf{X}) - \log Q_{\phi}(\mathbf{Z}) \bigg] &= \nabla_{[\theta, \phi]} \mathbb{E}_{Q(\mathbf{\epsilon})} \bigg[\log P_{\theta}(G_{\phi}(\epsilon), \mathbf{X}) - \log Q_{\phi}(G_{\phi}(\epsilon)) \bigg] \\
&= \mathbb{E}_{Q(\mathbf{\epsilon})} \bigg[\nabla_{[\theta, \phi]} \bigg( \log P_{\theta}(G_{\phi}(\mathbf{\epsilon}), \mathbf{X}) - \log Q_{\phi}(G_{\phi}(\epsilon)) \bigg) \bigg]
\tag{5}
\end{align}

Where $Q(\epsilon)$ is an independent source of randomness. Computing this expectation with empirical average, just like Equation (2), gives us a better estimate of the true gradient of ELBO (variance is reduced).

### II. Rao-Blackwellization

This is another well-known variance reduction technique. It is a bit mathematically rigorous, so we explain it simply without making it confusing. This requires the full variational distributions to have some kind of factorization. A specific case is when we have mean-field assumption, i.e.,

$Q_\phi(Z) = \prod_i Q_{\phi_i} (Z_i)$

With a little effort, we can pull out the gradient estimator for each of these $\phi_i$ parameters from (2). They look something like this:

\begin{equation}
\nabla_{\phi_i} \mathrm{ELBO}(\theta, \phi) = \mathbb{E}_{\mathbb{Q}_{\phi}} \bigg[ \nabla_{\phi_i} \log\mathbb{Q}_{\phi_i}(Z_i) \cdot \bigg( \overline{\log \mathbb{P}_{\theta}(\mathbf{Z}, \mathbf{X}) - \log \mathbb{Q}_{\phi}(\mathbf{Z})} \bigg)
+\cdots \bigg]
\tag{6}
\end{equation}

The reason why the quantity under the bar still has all the factors is because it is immune to gradient operator. Also because the expectation is outside the gradient operator, it contains all factors. At this point, the Rao-Blackwellization offers a variance-reduced estimate of the above gradient, i.e.,

\begin{equation}
\nabla_{\phi_i} \mathrm{ELBO}(\theta, \phi) \approx \mathbb{E}_{\mathbb{Q}_{\phi}^{(i)}} \bigg[ \nabla_{\phi_i} \log\mathbb{Q}_{\phi_i}(Z_i) \cdot \bigg( \overline{\log \mathbb{P}^{(i)}_{\theta}(\mathbf{Z}^{(i)}, \mathbf{X}) - \log \mathbb{Q}_{\phi_i}(Z_i)} \bigg)
+\cdots \bigg]
\tag{7}
\end{equation}

where $Z^{(i)}$ is the set of variables that forms the **markov blanket** of $Z_i$ with respect to the structure of the guide, $Q^{(i)}_\phi$ is the part of the variational distribution that depends on $Z^{(i)}$ and $P^{(i)}_\theta(Z^{(i)},\cdot)$ is the factor of the model that involves $Z^{(i)}$.

### III. Explicit enumeration for Discrete RVs

While exploiting the graph structure of the guide while simplifying (1), we might end up getting a term like this due to factorization in the guide density

$\mathbb{E}_{Z_i\sim\mathbb{Q}_{\phi_i}(Z_i)} \bigl[ f(\cdot) \bigr]$

If it happens that the variable $Z_i$  is discrete with the size of its state space reasonably small (e.g., a $d=5$ dimensional binary random variable has $2^5 = 32$ states), we can replace sampling-based empirical expectations with true expectation where we have to evaluate a sum over its entire state-space:

$\sum_{Z_i} \mathbb{Q}_{\phi_i}(Z_i)\cdot f(\cdot)$

So make sure the state-space is reasonable in size. This helps reducing the variance quite a bit.

This a lot pf math, but the good thing is that we hardly ever have to think about them in detail because they are accessible through libraries. One of them we are going to have a brief look on.


## Pyro: Universal Probabilistic Programming

Pyro is a probabilistic programming framework that allows users to write flexible models in terms of a simple API. Pyro is written in Python and uses the popular PyTorch library for its internal representation of computation graph and as auto differentiation engine. Pyro is quite expressive due to the fact that it allows the model/guide to have fully imperative flow. Its core API consists of these functionalities:

* `pyro.param()` for defining learnable parameters.
* `pyro.dist` contains a large collection of probability distribution.
* `pyro.sample()` for sampling from a given distribution.

Let us take a concrete example and work it out.

### Problem: Gaussian Mixture Model

The mixture of Gaussian is a relatively simple but widely studied probabilistic model. It has an important application in soft-clustering. For the sake of simplicity, we assume we only have two Gaussians. The generative view of the model is basically this: we flip a coin (latent) with a probability $\rho$ of getting the face up and, depending on the outcome $C \in \{0,1\}$, we sample data from $\mathcal{N}(\mu_0,\sigma_0)$ or $\mathcal{N}(\mu_1,\sigma_1)$:

$ C_i \sim Bernoulli(\rho)$ <P>
$X_i \sim \mathcal{N}(\mu_{C_i}, \sigma_{C_i})$

where $i=1 \dots N$ is the data index, $\theta \triangleq \{\rho,\mu_0,\sigma_0,\mu_1,\sigma_1\}$ is the set of model parameters we need to learn. This is how you write the model in Pyro:

In [ ]:
import pyro, torch, numpy as np
import pyro.distributions as dist
import pyro.optim         as optim
import pyro.infer         as infer
import matplotlib.pyplot  as plt
from   scipy.stats        import norm

plt.style.use('ggplot')
plt.ioff()

In [ ]:
@infer.config_enumerate(default='parallel')
def model(data): # Get the observed data as input
    # Define the coin probability as a parameter
    rho = pyro.param(
        "rho",                                    # Parameter name 
        torch.tensor([0.5]),                      # Initial value is 0.5
        constraint=dist.constraints.unit_interval # Has to be in range [0, 1]
        )
    # Define 2 means and 2 standard deviations with random initial values
    mean   = pyro.param("M", torch.tensor([1.5, 3.]))
    stddev = pyro.param("S", torch.tensor([0.5, 0.5]), constraint=dist.constraints.positive)
    
    with pyro.plate("data", len(data)):        # Define independence among variables
        rho_p = dist.Bernoulli(rho) 
        c    = pyro.sample("c", rho_p)          # c in {0, 1}
        c    = c.type(torch.LongTensor)
        X    = dist.Normal(mean[c], stddev[c]) # pick a mean for each 'c'
        x    = pyro.sample("x", X, obs=data)   # sample data and mark it as observed

Due to the discrete and low dimensional nature of the latent variable $C$, this problem is in general tractable in terms of computing posterior. But we will assume it is not. The true posterior $P(C_i \mid X_i)$ is the quantity known as "assignment" that reveals the latent factor, i.e., what was the coin toss result when a given $X_i$ was sampled. We define a guide on $C$, parameterized by variational parameters $\phi \triangleq \{ \lambda_i \}_{i=1}^N$ as:

$C_i \sim Bernoulli(\lambda_i)$

Next Pyro code defines such a guide.

In [ ]:
@infer.config_enumerate(default='parallel')
def guide(data): # Guide does not require data, it just needs the value of N
    phi = pyro.param("phi", torch.rand(len(data)), constraint=dist.constraints.unit_interval)
    with pyro.plate("data", len(data)):
        C = dist.Bernoulli(phi)
        c = pyro.sample("c", C)

We generate some synthetic data from the following simualator to train our model.

In [ ]:
def generate_data(N, mean1=3.0, mean2=-2.0, std1=1.0, std2=1.0):
    D1 = np.random.randn(N//2,) * std1 + mean1
    D2 = np.random.randn(N//2,) * std2 + mean2
    D  = np.concatenate([D1, D2], 0)
    np.random.shuffle(D)
    return torch.from_numpy(D.astype(np.float32))

Finally, Pyro requires a bit of boilerplate to setup the optimization. 

We plot:
* the ELBO loss
* the variational parameter 'lambda_i' for every data point
* the two gaussians in the model
* the coin probability 'rho' as the training progresses.

In [ ]:
mean1 = 2.0
mean2 = -1.0
std1  = 0.5
std2  = 0.5
data  = generate_data(200, mean1, mean2, std1, std2)

In [ ]:
pyro.render_model(
    model, 
    model_args=(data),
    render_distributions=True,
    render_params=True
)

In [ ]:
%matplotlib

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
  
pyro.clear_param_store()

optim   = pyro.optim.Adam({})
svi     = pyro.infer.SVI(model, guide, optim, infer.TraceEnum_ELBO())

losses = []
T      = 10000

for t in range(T):
    loss = svi.step(data)
    losses.append(loss)
    
    if t % 50 == 49:
        #print(f"step: {t}, ELBO loss: {loss :.2f}")
    
        ax[0].plot(losses, color='m')
        ax[0].scatter(len(losses), losses[-1], color='m')
        ax[0].annotate(f'{losses[-1]:.2f}', (len(losses)+50, losses[-1]+50))
        ax[0].set_xlabel("epochs")
        ax[0].set_ylabel("ELBO")
        ax[0].set_xlim([0, T])
        ax[0].set_ylim([0, 2500])

        phi = pyro.param("phi")

        han = ax[1].scatter(
            data.detach().numpy(), 
            phi.detach().numpy(), 
            c=phi.detach().numpy()
        )
        ax[1].set_ylim([-0.03, 1.03])
        ax[1].set_xlabel("Data axis")
        ax[1].set_ylabel(r"Posterior ($\mu_0, \mu_1, \sigma_0, \sigma_1$)")

        mean           = pyro.param("M")
        stddev         = pyro.param("S")
        coin_rho       = pyro.param("rho").detach().item()
        mean1e, mean2e = mean.detach().numpy()
        std1e, std2e   = stddev.detach().numpy()
        
        xmin, xmax     = data.min(), data.max()
        xs  = np.linspace(xmin-2, xmax+2, 150)
        y1  = norm.pdf(xs, mean1e, std1e)
        y2  = norm.pdf(xs, mean2e, std2e)
        p1, = ax[2].plot(xs, y1, color='r')
        p2, = ax[2].plot(xs, y2, color='b')
        ax[2].axvline(mean1e, linestyle='--', color='r')
        ax[2].axvline(mean2e, linestyle='--', color='b')
        cb  = ax[2].axhline(coin_rho, linestyle='--', color='black')
        ax[2].set_xlim([xmin-2, xmax+2])
        ax[2].set_ylim([-0.02, 0.8])
        ax[2].legend([p1, p2, cb], ['Gaussian 1', 'Gaussian 2', 'Coin Bias'], loc=2)
        ax[2].scatter(data.numpy(), np.zeros_like(data.numpy()), marker='x', c=phi.detach().numpy())
        ax[2].set_xlabel("Data axis")
        ax[2].set_ylabel("Model densities")

        # plt.draw()
        # plt.savefig(f"tmp/{t}.png", bbox_inches='tight', inches=0)
        
        plt.pause(0.01)
        ax[0].cla()
        ax[1].cla()
        ax[2].cla()

In [ ]:
print(f'True means:     {mean1  :.4f}  {mean2  :.4f}')
print(f'Inferred means: {mean1e :.4f}  {mean2e :.4f}')